In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
from nca.model import *
from nca.rollout import *
from nca.viz import *
from nca.stats import *

import torch
import numpy as np
import imageio
from IPython.display import Image, display
from IPython.display import Video

In [10]:
device = torch.device("cpu")
model = NCA()
state = torch.load("../../snapshots/froggy_model_noregen.pth", map_location=device)
model.load_state_dict(state)
model.eval()  # VERY IMPORTANT

NCA(
  (net): Sequential(
    (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
  )
)

In [11]:
x0 = seed(1)  # initial seed, shape [1, C, H, W]
with torch.no_grad():
    states = rollout(model, x0, 100, None, None, True, True)
    states = rollout(
        model,
        x0,
        steps=800,
        ablate_channel=None,
        width_cut=None,
        killEye=150,
        prune=False,
        realEyeReplace=None,
        realEyeCapture=0,
        fakeEyeReplace=None,
        inhibit=None
    )

In [13]:
frames = [viz.frame_to_rgb_uint8(f[:4]) for f in states] # Select first 4 channels (RGBA) and convert

save_crisp_mp4(
    frames,
    output_file="output.mp4",
    fps=60,
    upscale=1,
    frames_dir="frames", 
    quiet=True
)

# local file
Video("output.mp4", embed=True, width=400, height=400)

In [6]:
gif_frames = [frame_to_rgb_uint8(f[:4]) for f in states] # Select first 4 channels (RGBA) and convert

imageio.mimsave(
    "frog_killEye.gif",
    gif_frames,
    fps=30,
    loop=0  # infinite loop
)

display(Image(filename="frog_killEye.gif"))

NameError: name 'frame_to_rgb_uint8' is not defined

In [6]:
with imageio.get_writer(
    "frog_killEye2.mp4",
    fps=60,
    codec="libx264",
    quality=10,
    pixelformat="yuv444p",   # <- critical
    macro_block_size=None
) as writer:
    for frame in gif_frames:
        writer.append_data(frame)

In [7]:
make_crisp_mp4(gif_frames, "goof.mp4")

In [16]:
cell_states = get_states_by_part(states)
stats = compute_mean_var_by_part(cell_states)

# plot_means_by_parts(cell_states, stats)
# plot_variances_by_part(stats)
# plot_alive_vs_r_eye_difference(stats)

In [11]:
import subprocess
import os

os.makedirs("frames", exist_ok=True)

# Save frames as PNG first
for i, frame in enumerate(gif_frames):
    frame_path = f"frames/frame_{i:04d}.png"
    imageio.imwrite(frame_path, frame)

# Build the ffmpeg command
cmd = [
    "ffmpeg",
    "-framerate", "60",           # playback speed
    "-i", "frames/frame_%04d.png",# input frames
    "-c:v", "libx264",            # H264 codec
    "-pix_fmt", "yuv444p",        # keep pixel-perfect colors
    "-crf", "0",                  # lossless
    "-preset", "veryslow",        # better compression
    "frog_killEye.mp4"            # output
]

subprocess.run(cmd, check=True)

ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20260103
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enab

CompletedProcess(args=['ffmpeg', '-framerate', '60', '-i', 'frames/frame_%04d.png', '-c:v', 'libx264', '-pix_fmt', 'yuv444p', '-crf', '0', '-preset', 'veryslow', 'frog_killEye.mp4'], returncode=0)

In [12]:
# local file
Video("frog_killEye.mp4", embed=True, width=400, height=400)